In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from ugdatalab.models.galaxy_zoo import GalaxyZooDataset
from ugdatalab.models.galaxy_zoo.constants import N_LABELS
from ugdatalab.methods.cnn import train_cnn, count_parameters, rmse_loss
from ugdatalab.methods.architectures import build_resnet18

import plotters

# Galaxy Image Classification — ResNet-18

## Task 14 — Residual Networks

**Residual Networks** (He et al. 2015) address the *degradation problem*: as traditional CNNs are made deeper by stacking more layers, training accuracy saturates and then degrades — not because of overfitting (training error increases too) but because deeper networks are harder to optimize. The key insight is that learning the *residual* $F(x) = H(x) - x$ is easier than learning the full mapping $H(x)$ directly, where $x$ is the input to a block and $H(x)$ is the desired output.

Each **residual block** computes $H(x) = F(x) + x$ via a *shortcut connection* (or *skip connection*) that adds the input $x$ directly to the output of two or three stacked convolutional layers. If the optimal transformation is close to the identity (i.e., the block should mostly "pass through" its input), the network only needs to learn the small perturbation $F(x) \approx 0$, which is much easier than learning $H(x) \approx x$ from scratch.

This differs from traditional deep CNNs in two critical ways:
1. **Gradient flow**: skip connections provide a direct path for gradients during backpropagation, mitigating the vanishing gradient problem.
2. **Deeper = better**: with skip connections, adding more layers never hurts — the extra layers can always learn the identity and reduce to a shallower network.

We use the 18-layer variant (ResNet-18), modified to accept our downsized $69 \times 69$ input and output 37 sigmoid-activated labels.

In [ ]:
# Load preprocessed data
img_data = np.load("galaxy_zoo_images.npz")
images = img_data["images"]
galaxy_ids = img_data["galaxy_ids"]

label_data = np.load("galaxy_zoo_labels.npz")
labels = label_data["labels"]

split_data = np.load("split_indices.npz")
train_idx = split_data["train_idx"]
val_idx = split_data["val_idx"]

train_images = images[train_idx]
val_images = images[val_idx]
train_labels = labels[train_idx]
val_labels = labels[val_idx]

TARGET_SIZE = images.shape[1]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Train: {train_images.shape}, Val: {val_images.shape}")

## Task 15 — Train ResNet-18

In [ ]:
train_ds = GalaxyZooDataset(train_images, train_labels, transform=None)
val_ds = GalaxyZooDataset(val_images, val_labels, transform=None)

model = build_resnet18(n_labels=N_LABELS, input_size=TARGET_SIZE)
print(f"ResNet-18 trainable parameters: {count_parameters(model):,}")

resnet_result = train_cnn(
    model=model,
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=64,
    n_epochs=30,
    lr=1e-3,
    device=DEVICE,
    seed=42,
    scheduler_factory=None,
    num_workers=4,
)
print(f"\nBest epoch: {resnet_result.best_epoch + 1}")
print(f"Best validation RMSE: {resnet_result.best_val_loss:.4f}")

In [ ]:
ax = plotters.plot_loss_curves(
    resnet_result.train_losses, resnet_result.val_losses, "ResNet-18",
)
plt.show()

In [ ]:
# Save model and results
torch.save(resnet_result.model_state, "resnet18.pt")
np.savez_compressed(
    "resnet_result.npz",
    train_losses=resnet_result.train_losses,
    val_losses=resnet_result.val_losses,
    best_epoch=resnet_result.best_epoch,
    best_val_loss=resnet_result.best_val_loss,
    n_parameters=resnet_result.n_parameters,
    learning_rates=resnet_result.learning_rates,
)
print("Saved resnet18.pt and resnet_result.npz")